# Error Analysis & WER Robustness Check

Run this **after** you've evaluated a model on the 54-file held-out set and saved a results CSV (same pattern as your Baseline WER notebook: columns `filename`, `intended_text`, `predicted_text`).

This notebook answers three questions your single overall-WER number can't:

1. **Is the difference between your runs (23.7% / 23.69% / 19.79%) a real effect, or noise?** — with only 54 held-out files, one flipped example is worth ~1.85 WER points. A bootstrap confidence interval tells you how much to trust any single number.
2. **Is pooled ("micro") WER being dominated by your longer sentences?** — a 12-word sentence with 3 errors contributes far more to a pooled edit-distance/word-count ratio than a single missed word does. Comparing micro vs. macro WER shows whether that's happening.
3. **Which item types / categories are actually driving your errors**, so your next data-collection pass is targeted instead of a guess.

Point `RESULTS_CSV_PATH` below at whichever results file you want to analyze (defaults to your v2 LoRA results).

## Step 1 - Connect to Google Drive and set paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os

base_path = "/content/drive/MyDrive/capstone-cleft-speech-data"

# Point this at the results CSV you want to analyze, e.g.:
#   f"{base_path}/tracking/results_LORA_finetuned.csv"   (v2)
#   f"{base_path}/tracking/results_LORA_v3.csv"          (v3, once you've evaluated it)
RESULTS_CSV_PATH = f"{base_path}/tracking/results_LORA_finetuned.csv"

csv_path = f"{base_path}/tracking/dataset.csv"
print("Analyzing:", RESULTS_CSV_PATH)

Mounted at /content/drive
Analyzing: /content/drive/MyDrive/capstone-cleft-speech-data/tracking/results_LORA_finetuned.csv


## Step 2 - Recreate the exact same held-out split

Same `random_state=42`, `test_size=0.15` as every other notebook in this project — this is what guarantees you're looking at the real held-out set, not accidentally including training files.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(csv_path)
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
val_filenames = set(val_df['filename'])

results_df = pd.read_csv(RESULTS_CSV_PATH)
val_results = results_df[results_df['filename'].isin(val_filenames)].copy()
print("Held-out files found in results CSV:", len(val_results), "/", len(val_filenames))

Held-out files found in results CSV: 54 / 54


## Step 3 - Clean text (same normalization as your baseline notebook) and compute per-file WER

In [5]:
!pip install jiwer
import re
import jiwer

def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    return text

val_results["intended_clean"] = val_results["intended_text"].apply(clean_text)
val_results["predicted_clean"] = val_results["predicted_text"].apply(clean_text)

val_results["file_wer"] = val_results.apply(
    lambda row: jiwer.wer(row["intended_clean"], row["predicted_clean"]) if row["intended_clean"].strip() else 0.0,
    axis=1,
)
val_results["n_words"] = val_results["intended_clean"].apply(lambda t: len(t.split()))

val_results.sort_values("file_wer", ascending=False).head(10)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 47.2 MB/s eta 0:00:00


,filename,intended_text,predicted_text,intended_clean,predicted_clean,wer,file_wer,n_words
224,achieves3.mp3,achieves,accuse,achieves,accuse,1.000000,1.000000,1
126,snake.mp3,snake,lake,snake,lake,1.000000,1.000000,1
356,today weather.mp3,Today in Jadcherla it’s cloudy and humid with ...,"Today, in Jarkala, it is a cloudy and humid wi...",today in jadcherla its cloudy and humid with a...,today in jarkala it is a cloudy and humid with...,0.661538,0.661538,65
354,1-20.mp3,"one, two, three, four, five, six, seven, eight...","one, two, three, four, five, six, seven, eight...",one two three four five six seven eight nine t...,one two three four five six seven eight nine t...,0.500000,0.500000,40
351,morning routines.mp3,"This morning I woke up at 6:30, freshened up a...","This morning, I woke up at 6.30am, freshened u...",this morning i woke up at 630 freshened up and...,this morning i woke up at 630am freshened up a...,0.057692,0.057692,52
5,pencil3.mp3,pencil,pencil,pencil,pencil,0.000000,0.000000,1
33,tiger.mp3,tiger,tiger,tiger,tiger,0.000000,0.000000,1
39,ticket.mp3,ticket,ticket,ticket,ticket,0.000000,0.000000,1
42,water.mp3,water,water,water,water,0.000000,0.000000,1
45,doctor.mp3,doctor,doctor,doctor,doctor,0.000000,0.000000,1


## Step 4 - Micro vs. macro WER

- **Micro WER** = total word edits / total reference words across the whole set (this is what `jiwer.wer()` on the full lists gives you, and almost certainly what you've been reporting as "23.7%"/"23.69%").
- **Macro WER** = average of each file's own WER, unweighted by sentence length.

If these two numbers are noticeably different, it means your longer items (sentences) are pulling the pooled number around more than a per-item average would suggest — useful context for how you describe the result in your report/paper.

In [6]:
micro_wer = jiwer.wer(list(val_results["intended_clean"]), list(val_results["predicted_clean"]))
macro_wer = val_results["file_wer"].mean()

print(f"Micro WER (pooled, standard): {micro_wer*100:.3f}%")
print(f"Macro WER (mean of per-file WER): {macro_wer*100:.3f}%")
print(f"Difference: {abs(micro_wer-macro_wer)*100:.3f} points")

Micro WER (pooled, standard): 23.693%
Macro WER (mean of per-file WER): 5.962%
Difference: 17.732 points


## Step 5 - Bootstrap confidence interval on held-out WER

Resamples your 54 held-out files with replacement 5,000 times and recomputes micro-WER each time. The resulting spread tells you how much a WER number like "23.69%" can be expected to move around just from which 54 files happened to land in your held-out set — this is the number to compare 19.79% / 21.875% / 23.693% against before concluding one run is genuinely better than another.

In [7]:
import numpy as np

np.random.seed(42)
n_boot = 5000
n = len(val_results)
refs = val_results["intended_clean"].tolist()
hyps = val_results["predicted_clean"].tolist()

boot_wers = []
for _ in range(n_boot):
    idx = np.random.randint(0, n, size=n)
    sample_refs = [refs[i] for i in idx]
    sample_hyps = [hyps[i] for i in idx]
    boot_wers.append(jiwer.wer(sample_refs, sample_hyps))

boot_wers = np.array(boot_wers) * 100
print(f"Point estimate (micro WER): {micro_wer*100:.3f}%")
print(f"Bootstrap median: {np.median(boot_wers):.3f}%")
print(f"95% CI: [{np.percentile(boot_wers, 2.5):.3f}%, {np.percentile(boot_wers, 97.5):.3f}%]")
print()
print("If two runs' WER numbers both fall inside each other's CI, treat them as")
print("statistically indistinguishable given this held-out set size, not as a real difference.")

Point estimate (micro WER): 23.693%
Bootstrap median: 22.388%
95% CI: [1.463%, 42.201%]

If two runs' WER numbers both fall inside each other's CI, treat them as
statistically indistinguishable given this held-out set size, not as a real difference.


## Step 6 - WER by item type (single word vs. sentence)

Uses word count of the reference text as a proxy for item type (1 word = word-list item, otherwise a sentence). If your errors cluster heavily in one bucket, that tells you where to focus — e.g. if sentence-level WER is much worse, the issue may be more about longer-context decoding than individual phoneme recognition.

In [8]:
val_results["item_type"] = val_results["n_words"].apply(lambda n: "single_word" if n <= 1 else "sentence")

for item_type, group in val_results.groupby("item_type"):
    grp_wer = jiwer.wer(list(group["intended_clean"]), list(group["predicted_clean"]))
    print(f"{item_type}: n={len(group)}, micro WER={grp_wer*100:.3f}%")

sentence: n=14, micro WER=26.721%
single_word: n=40, micro WER=5.000%


## Step 7 - WER by phoneme category (optional — needs your Recording Tracker)

Your Recording Tracker spreadsheet has a `category` column (the 8 phoneme categories: p/b, t/d, k/g, s/z, sh/ch/j, m/n, l/r, f/v) that `dataset.csv` doesn't carry. Point `TRACKER_PATH` at it and set `FILENAME_COL`/`CATEGORY_COL` to match its actual column names. If the file isn't found or the columns don't match, this cell will tell you what it saw instead of failing silently — adjust and re-run.

In [9]:
TRACKER_PATH = f"{base_path}/tracking/Recording_Tracker.xlsx"
FILENAME_COL = "filename"
CATEGORY_COL = "category"

if os.path.exists(TRACKER_PATH):
    tracker_df = pd.read_excel(TRACKER_PATH)
    print("Tracker columns found:", list(tracker_df.columns))

    if FILENAME_COL in tracker_df.columns and CATEGORY_COL in tracker_df.columns:
        merged = val_results.merge(
            tracker_df[[FILENAME_COL, CATEGORY_COL]],
            left_on="filename", right_on=FILENAME_COL, how="left"
        )
        print()
        print("Held-out files with a matched category:", merged[CATEGORY_COL].notna().sum(), "/", len(merged))
        print()
        cat_report = []
        for category, group in merged.groupby(CATEGORY_COL):
            if len(group) == 0:
                continue
            cat_wer = jiwer.wer(list(group["intended_clean"]), list(group["predicted_clean"]))
            cat_report.append({"category": category, "n_files": len(group), "wer_pct": round(cat_wer*100, 2)})
        cat_report_df = pd.DataFrame(cat_report).sort_values("wer_pct", ascending=False)
        display(cat_report_df)
    else:
        print(f"Could not find columns '{FILENAME_COL}' / '{CATEGORY_COL}' in the tracker.")
        print("Update FILENAME_COL / CATEGORY_COL above to match the actual column names shown above and re-run.")
else:
    print(f"No file found at {TRACKER_PATH}. Update TRACKER_PATH to your Recording Tracker's actual location and re-run.")
    print("Skipping category breakdown - Step 8 below still works without it.")

No file found at /content/drive/MyDrive/capstone-cleft-speech-data/tracking/Recording_Tracker.xlsx. Update TRACKER_PATH to your Recording Tracker's actual location and re-run.
Skipping category breakdown - Step 8 below still works without it.


## Step 8 - Full error listing, worst-first

Every held-out file sorted by its own WER, worst to best, with intended vs. predicted text side by side. This is the manual-inspection step — read through the worst rows and look for a repeating pattern (a specific phoneme, cluster, or word position) rather than treating each error as independent. That pattern is what should drive your next round of targeted recordings.

In [10]:
pd.set_option("display.max_colwidth", None)
error_view = val_results[val_results["file_wer"] > 0].sort_values("file_wer", ascending=False)
error_view[["filename", "intended_text", "predicted_text", "file_wer", "item_type"]]

,filename,intended_text,predicted_text,file_wer,item_type
126,snake.mp3,snake,lake,1.000000,single_word
224,achieves3.mp3,achieves,accuse,1.000000,single_word
356,today weather.mp3,"Today in Jadcherla it’s cloudy and humid with a temperature around 23°C. There’s a high chance of showers through the afternoon and evening, with rain likely picking up later at night. Expect about 0.83 inches of rain for the day and humidity staying around 96%. The air quality is good. Overall it’s a cool, wet day so keep an umbrella handy if you’re heading out.","Today, in Jarkala, it is a cloudy and humid with a temperature around 23 degrees Celsius. There is a high chance of showers through the afternoon and evening, which rained lightly picking of nature at night, especially about 0.83 inches of rain.",0.661538,sentence
354,1-20.mp3,"one, two, three, four, five, six, seven, eight, nine, ten, eleven, twelve, thirteen, fourteen, fifteen, sixteen, seventeen, eighteen, nineteen, twenty, one, two, three, four, five, six, seven, eight, nine, ten, eleven, twelve, thirteen, fourteen, fifteen, sixteen, seventeen, eighteen, nineteen, twenty","one, two, three, four, five, six, seven, eight, nine, ten, eleven, twelve, thirteen, fourteen, fifteen, sixteen, seventeen, eighteen, nineteen, twenty,",0.500000,sentence
351,morning routines.mp3,"This morning I woke up at 6:30, freshened up and went to the temple. After that I recorded some voice notes for my project. Then I attended all my classes. Later I had my lunch - lemon rice, it was really good and spent some time relaxing before getting back to the work.","This morning, I woke up at 6.30am, freshened up and went to the temple. After that, I recorded some voice notes for my project. Then I attended all my classes. Later, I had my lunch — 11 right. It was really good, and spent some time relaxing before getting back to the work.",0.057692,sentence


## Step 9 - Save the analysis summary

In [11]:
summary_path = f"{base_path}/tracking/error_analysis_summary.csv"
val_results.to_csv(summary_path, index=False)
print("Saved per-file breakdown to:", summary_path)
print()
print(f"Micro WER: {micro_wer*100:.3f}%  |  Macro WER: {macro_wer*100:.3f}%")
print(f"Bootstrap 95% CI: [{np.percentile(boot_wers, 2.5):.3f}%, {np.percentile(boot_wers, 97.5):.3f}%]")

Saved per-file breakdown to: /content/drive/MyDrive/capstone-cleft-speech-data/tracking/error_analysis_summary.csv

Micro WER: 23.693%  |  Macro WER: 5.962%
Bootstrap 95% CI: [1.463%, 42.201%]


## What to do with these results

- If the bootstrap CI is wide (likely, given n=54) and your different runs' WER numbers all fall inside it, don't report them as meaningfully different — report the CI itself, it's more defensible in your paper/viva than a bare point estimate.
- If one item type or phoneme category clearly has the worst WER, that's your target for the next round of real recordings (Section 4 of your project brief — same recording protocol, just weighted toward the weak category) rather than another blind hyperparameter sweep.
- Bring this notebook's output back and we can decide the next concrete experiment together.